In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_OpenMM_Available_platforms)=
# Available platforms

*Querying and inspecting available hardware compute platforms in OpenMM.*

The function {func}`molsysmt.third_party.openmm.platforms.available_platforms` lists registered OpenMM platform backends on the host machine. Names are case-sensitive and can include external plugins. Registration does not initialize a device or guarantee that it can create a calculation context.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects: {func}`molsysmt.third_party.openmm.platforms.available_platforms`.
:::


## Basic usage

Let's inspect available OpenMM platforms on the local machine:

In [2]:
import molsysmt as msm

msm.third_party.openmm.platforms.available_platforms()

Platform Reference with speed 1.0
Platform CPU with speed 10.0
Platform CUDA with speed 100.0


## Inspecting registration

The default call prints names and engine speed estimates. Set `verbose=False` to receive a list of registered names. This metadata query performs no numerical calculation or device-readiness test.

In [3]:
registered = msm.third_party.openmm.platforms.available_platforms(verbose=False)
print('Registered platform names:', registered)

Registered platform names: ['Reference', 'CPU', 'CUDA']


## Selecting an execution platform

The mechanics API accepts a platform name in the selected engine's vocabulary. The OpenMM adapter resolves the exact name and constructs on that platform. This small bundled system demonstrates `Reference` without requiring GPU hardware.

In [4]:
molsys = msm.convert(msm.systems['alanine dipeptide']['alanine_dipeptide.h5msm'], structure_indices=0)
context = msm.convert(molsys, to_form='openmm.Context', platform='Reference')
print('Actual context platform:', context.getPlatform().getName())
energy = msm.molecular_mechanics.get_potential_energy(context)
print('Potential energy:', energy)
del context

Actual context platform: Reference
Potential energy: -55.79426034236738 kilojoule / mole


:::{note}
:class: dropdown

**Demo Systems Catalog**

This example uses bundled alanine dipeptide. See {ref}`Demo Systems <user-foundations-entrance-demo-systems>`.
:::

## Understanding failures

| Observation | Outcome |
| --- | --- |
| Empty or non-string name | Native argument error. |
| Name unknown to the OpenMM registry/vocabulary | Native choice diagnostic. |
| Known OpenMM name absent from registration | `ExecutionPlatformUnavailableError`, with registered names. |
| Engine package absent | Native dependency diagnostic. |
| Registered platform fails driver/device/context initialization | Original engine execution error. |

A failed explicit choice does not switch to another platform. Installed plugin names are accepted dynamically. The known engine vocabulary includes Reference, CPU, CUDA, OpenCL, HIP and Metal; listing a name does not advertise successful execution on its hardware.

`platform` controls new contexts. An existing Context/Simulation keeps its actual platform, even when a general mechanics call has a different default. Platform precision and settings can affect numerical agreement; this contract does not promise bitwise identity across platforms. See [OpenMM platform properties](https://docs.openmm.org/latest/userguide/library/04_platform_specifics.html).

:::{seealso}
:class: dropdown

- {ref}`openmm.Simulation <Tutorial_Form_openmm_Simulation>`: OpenMM Simulation form adapter in MolSysMT.
- {ref}`openmm.System <Tutorial_Form_openmm_System>`: OpenMM System container in MolSysMT.
- {ref}`Convert <Tutorial_Convert>`: Converting molecular systems with {func}`molsysmt.basic.convert`.
:::